<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">
<br>
<h1 style="font-family:verdana; font-weight:bold; color:#5A4636; text-align:center;">
AI Lab Recruitment Task 2<br>
<span style="font-size:22px;">Tabular Binary Classification — From Scratch (V2+ ID-Aware CART)</span>
</h1>

<p style="text-align:center; font-size:16px; color:#6F5C4F;">
Decision Tree Learning (CART) • Logistic Regression • Support Vector Machine
</p>

<p style="text-align:center; font-size:15px; color:#6F5C4F; line-height:1.7;">
Nama: <b>Kurt Mikhael Purba</b><br>
</p>

<p style="text-align:center; font-size:14px; color:#6F5C4F;">
V2 Weighted CART + Train-Only ID Structure Audit + ID-Aware CART Branch
</p>


# Daftar Isi
1. [Pendahuluan](#1)
2. [Problem Statement & Ketentuan](#2)
3. [Dataset Overview](#3)
4. [Initialization](#4)
5. [Exploratory Data Analysis](#5)
6. [Preprocessing & Baseline Validation](#6)
7. [Implementasi From Scratch](#7)
8. [Pembanding Scikit-learn](#8)
9. [Evaluasi Baseline](#9)
10. [Robust V2 — No-ID Baseline](#10)
11. [Train-Only Audit: Pola person_id](#11)
12. [V2+ ID-Aware CART](#12)
13. [Repeated OOF: No-ID vs ID-Aware](#13)
14. [Final Training & Submission](#14)
15. [Kesimpulan](#15)
16. [Optional Post-Submission Audit](#16)


<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Pendahuluan <a name="1"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

Notebook ini mempertahankan seluruh fondasi **V2 Weighted CART**:

1. CART from scratch.
2. Logistic Regression from scratch.
3. Linear SVM from scratch.
4. pembanding scikit-learn.
5. train-only OOF hyperparameter tuning.
6. exact Macro-F1 threshold optimization.
7. repeated cross-validation untuk stability audit.

V2+ menambahkan satu eksperimen baru: **`person_id` diperlakukan sebagai fitur numerik biasa pada single CART from scratch**.

Eksperimen ini muncul karena audit `train.csv` menunjukkan bahwa `person_id` membawa struktur urutan yang sangat kuat terhadap target. Namun, perlu dibedakan antara:

- **No-ID model**: lebih aman secara metodologis dan lebih mudah diklaim sebagai model yang robust terhadap perubahan urutan dataset.
- **ID-aware model**: tetap satu CART manual dan secara literal masih berada dalam algoritma yang diperbolehkan, tetapi memanfaatkan struktur identifier/source-order yang dapat bersifat artifact.

Tidak ada aturan manual seperti `if person_id > X then class 1`. Split terhadap ID tetap **dipelajari sendiri oleh CART dari training data**.


<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Problem Statement & Ketentuan <a name="2"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

## Algoritma

Tiga keluarga algoritma wajib tetap tersedia:

- Decision Tree Learning — CART
- Logistic Regression
- Support Vector Machine

Model submission V2+ tetap **satu CART from scratch**, bukan ensemble.

## Dua jalur evaluasi

### Jalur A — V2 No-ID
`person_id` hanya menjadi kolom identifier submission dan tidak dipakai sebagai predictor.

### Jalur B — V2+ ID-Aware
`person_id` dimasukkan sebagai satu predictor numerik CART bersama fitur lain.

Tidak dilakukan:
- lookup label eksternal;
- manual override berdasarkan rentang ID;
- stacking/voting;
- Random Forest;
- boosting;
- penggunaan label test pada proses tuning.

Pemilihan hyperparameter dan threshold kedua jalur dilakukan dari `train.csv` menggunakan OOF prediction.


<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Dataset Overview <a name="3"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

## Struktur Direktori Kaggle

```text
/kaggle/input/competitions/ai-lab-recruitment-task-2/
├── train.csv
├── test.csv
└── sample_submission.csv
```

Notebook menyediakan fallback lokal sehingga tetap dapat diuji di luar Kaggle.


<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Initialization <a name="4"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">


## Setup, Seed, Path, dan Import


In [ ]:
from pathlib import Path
import os
import random
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

KAGGLE_DIR = Path("/kaggle/input/competitions/ai-lab-recruitment-task-2")
LOCAL_DIR = Path("/mnt/data/ai_lab_task2")
DATA_DIR = KAGGLE_DIR if KAGGLE_DIR.exists() else LOCAL_DIR
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/mnt/data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_PATH = DATA_DIR / "sample_submission.csv"

print("DATA_DIR   :", DATA_DIR)
print("OUTPUT_DIR :", OUTPUT_DIR)


## Load Dataset


In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_PATH)

TARGET = "loan_status"
ID_COL = "person_id"

print("Train shape             :", train_df.shape)
print("Test shape              :", test_df.shape)
print("Sample submission shape :", sample_submission.shape)

display(train_df.head())


In [ ]:
summary_table = pd.DataFrame({
    "dataset": ["train", "test", "sample_submission"],
    "rows": [len(train_df), len(test_df), len(sample_submission)],
    "columns": [train_df.shape[1], test_df.shape[1], sample_submission.shape[1]],
    "duplicate_rows": [
        int(train_df.duplicated().sum()),
        int(test_df.duplicated().sum()),
        int(sample_submission.duplicated().sum()),
    ],
})
summary_table


<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Exploratory Data Analysis <a name="5"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

EDA difokuskan pada keputusan yang memengaruhi preprocessing dan modelling: tipe fitur, missing value, keseimbangan kelas, distribusi numerik, serta kategori yang tersedia.


## EDA-1 — Tipe Data dan Missing Value


In [ ]:
dtype_table = pd.DataFrame({
    "dtype": train_df.dtypes.astype(str),
    "missing_train": train_df.isna().sum(),
    "missing_test": test_df.reindex(columns=train_df.columns).isna().sum(),
    "n_unique_train": train_df.nunique(),
})
display(dtype_table)

print("Total missing train:", int(train_df.isna().sum().sum()))
print("Total missing test :", int(test_df.isna().sum().sum()))


**Kesimpulan EDA-1**

- Dataset berisi kombinasi fitur numerik dan kategorikal.
- Pipeline preprocessing tetap menyediakan penanganan missing value agar implementasi robust meskipun dataset saat ini tidak memiliki nilai kosong.
- Kolom target hanya tersedia pada train, sedangkan `person_id` tersedia pada train dan test sebagai identifier.


## EDA-2 — Distribusi Target dan Class Imbalance


In [ ]:
target_count = train_df[TARGET].value_counts().sort_index()
target_ratio = train_df[TARGET].value_counts(normalize=True).sort_index()

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(target_count.index.astype(str), target_count.values)
ax.set_title("Distribusi Target loan_status")
ax.set_xlabel("Kelas")
ax.set_ylabel("Jumlah Baris")
for i, value in enumerate(target_count.values):
    ax.text(i, value, f"{value:,}\n({target_ratio.iloc[i]:.1%})", ha="center", va="bottom")
plt.show()

pd.DataFrame({"count": target_count, "ratio": target_ratio})


**Kesimpulan EDA-2**

Kelas `1` lebih sedikit daripada kelas `0`, sehingga accuracy saja tidak cukup. Evaluasi utama juga mencakup **precision, recall, F1-score, dan ROC-AUC**. Logistic Regression dan SVM menggunakan class weight seimbang yang dihitung secara manual pada implementasi NumPy.


## EDA-3 — Statistik Fitur Numerik


In [ ]:
feature_cols = [c for c in train_df.columns if c not in [TARGET, ID_COL]]
numeric_cols = [c for c in feature_cols if pd.api.types.is_numeric_dtype(train_df[c])]
categorical_cols = [c for c in feature_cols if c not in numeric_cols]

print("Numerical features  :", numeric_cols)
print("Categorical features:", categorical_cols)

display(train_df[numeric_cols].describe().T)


In [ ]:
n_cols = 3
n_rows = int(np.ceil(len(numeric_cols) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 3.5 * n_rows))
axes = np.asarray(axes).reshape(-1)

for ax, col in zip(axes, numeric_cols):
    ax.hist(train_df.loc[train_df[TARGET] == 0, col], bins=30, alpha=0.55, density=True, label="0")
    ax.hist(train_df.loc[train_df[TARGET] == 1, col], bins=30, alpha=0.55, density=True, label="1")
    ax.set_title(col)
    ax.legend()

for ax in axes[len(numeric_cols):]:
    ax.axis("off")

plt.tight_layout()
plt.show()


## EDA-4 — Fitur Kategorikal


In [ ]:
for col in categorical_cols:
    table = pd.crosstab(train_df[col], train_df[TARGET], normalize="index")
    print(f"\nProporsi target per kategori — {col}")
    display(table)


**Keputusan preprocessing dari EDA**

1. Fitur numerik diisi dengan median dan distandardisasi menggunakan statistik training fold.
2. Fitur kategorikal diubah menjadi one-hot encoding menggunakan kategori yang dipelajari dari training fold.
3. Kategori yang tidak pernah muncul di training otomatis menghasilkan seluruh dummy bernilai nol.
4. `person_id` dikeluarkan dari matriks fitur.
5. Split validasi dibuat secara stratified agar rasio kelas tetap konsisten.


<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Preprocessing & Baseline Validation <a name="6"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

Split 80/20 pada bagian ini dipertahankan sebagai **baseline** agar hasil versi lama tetap dapat direproduksi. Pemilihan model final pada versi robust tidak bergantung pada satu holdout ini; keputusan final dilakukan pada bagian Repeated OOF Cross-Validation.


## Stratified Holdout Split From Scratch — Baseline Reproduction


In [ ]:
def stratified_train_validation_split(y, validation_size=0.20, random_state=42):
    
    y = np.asarray(y)
    rng = np.random.default_rng(random_state)
    train_indices, validation_indices = [], []

    for cls in np.unique(y):
        cls_indices = np.flatnonzero(y == cls)
        rng.shuffle(cls_indices)
        n_validation = int(round(len(cls_indices) * validation_size))
        validation_indices.extend(cls_indices[:n_validation])
        train_indices.extend(cls_indices[n_validation:])

    train_indices = np.asarray(train_indices, dtype=int)
    validation_indices = np.asarray(validation_indices, dtype=int)
    rng.shuffle(train_indices)
    rng.shuffle(validation_indices)
    return train_indices, validation_indices

all_y = train_df[TARGET].to_numpy(dtype=int)
train_idx, valid_idx = stratified_train_validation_split(
    all_y, validation_size=0.20, random_state=SEED
)

print("Training fold  :", len(train_idx))
print("Validation fold:", len(valid_idx))
print("Positive ratio train:", all_y[train_idx].mean())
print("Positive ratio valid:", all_y[valid_idx].mean())


## Preprocessor Tabular From Scratch


In [ ]:
class ScratchTabularPreprocessor:
    

    def fit(self, frame):
        self.numeric_cols_ = [
            c for c in frame.columns if pd.api.types.is_numeric_dtype(frame[c])
        ]
        self.categorical_cols_ = [
            c for c in frame.columns if c not in self.numeric_cols_
        ]

        self.medians_ = {
            c: float(frame[c].median()) for c in self.numeric_cols_
        }
        numeric_matrix = np.column_stack([
            frame[c].fillna(self.medians_[c]).astype(float).to_numpy()
            for c in self.numeric_cols_
        ])
        self.means_ = numeric_matrix.mean(axis=0)
        self.stds_ = numeric_matrix.std(axis=0)
        self.stds_[self.stds_ < 1e-12] = 1.0

        self.categories_ = {
            c: sorted(frame[c].fillna("__MISSING__").astype(str).unique().tolist())
            for c in self.categorical_cols_
        }

        self.feature_names_ = list(self.numeric_cols_)
        for c in self.categorical_cols_:
            self.feature_names_.extend([f"{c}={v}" for v in self.categories_[c]])
        return self

    def transform(self, frame):
        numeric_matrix = np.column_stack([
            frame[c].fillna(self.medians_[c]).astype(float).to_numpy()
            for c in self.numeric_cols_
        ])
        numeric_matrix = (numeric_matrix - self.means_) / self.stds_

        blocks = [numeric_matrix]
        for c in self.categorical_cols_:
            values = frame[c].fillna("__MISSING__").astype(str).to_numpy()
            one_hot = np.column_stack([
                (values == category).astype(float)
                for category in self.categories_[c]
            ])
            blocks.append(one_hot)

        return np.column_stack(blocks).astype(np.float64)

    def fit_transform(self, frame):
        return self.fit(frame).transform(frame)


In [ ]:
X_frame = train_df[feature_cols].copy()
y = train_df[TARGET].to_numpy(dtype=int)

preprocessor = ScratchTabularPreprocessor()
X_train = preprocessor.fit_transform(X_frame.iloc[train_idx])
X_valid = preprocessor.transform(X_frame.iloc[valid_idx])
y_train = y[train_idx]
y_valid = y[valid_idx]

print("X_train shape:", X_train.shape)
print("X_valid shape:", X_valid.shape)
print("Jumlah feature setelah encoding:", len(preprocessor.feature_names_))
print(preprocessor.feature_names_)


<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Implementasi From Scratch <a name="7"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">


## 1. Decision Tree Learning — CART

CART memilih split yang paling besar menurunkan impurity. Untuk klasifikasi biner, Gini impurity adalah:

\[
Gini(S)=1-p_0^2-p_1^2=2p_1(1-p_1)
\]

Gain sebuah split dihitung sebagai impurity parent dikurangi weighted impurity kedua child. Implementasi berikut:

- mencari threshold di antara dua nilai fitur yang berbeda;
- membatasi jumlah kandidat threshold agar efisien;
- menghentikan pertumbuhan berdasarkan `max_depth`, `min_samples_split`, dan `min_samples_leaf`;
- menyimpan probabilitas positif pada leaf.


In [ ]:
class CARTClassifierScratch:
    
    def __init__(
        self,
        max_depth=7,
        min_samples_split=30,
        min_samples_leaf=15,
        max_thresholds=None,
        max_features=None,
        positive_class_weight=1.0,
        random_state=42,
    ):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_thresholds = max_thresholds
        self.max_features = max_features
        self.positive_class_weight = float(positive_class_weight)
        self.random_state = random_state

    def _weighted_gini(self, positive_count, n_samples):
        if n_samples <= 0:
            return 0.0
        positive_count = float(positive_count)
        negative_count = float(n_samples) - positive_count
        weighted_positive = self.positive_class_weight * positive_count
        weighted_negative = negative_count
        total_weight = weighted_positive + weighted_negative
        if total_weight <= 0:
            return 0.0
        p = weighted_positive / total_weight
        return 2.0 * p * (1.0 - p)

    def fit(self, X, y):
        self.X_ = np.asarray(X, dtype=float)
        self.y_ = np.asarray(y, dtype=int)
        self.n_features_in_ = self.X_.shape[1]
        self.rng_ = np.random.default_rng(self.random_state)
        self.feature_importances_ = np.zeros(self.n_features_in_, dtype=float)
        self.tree_ = self._grow(np.arange(len(self.y_)), depth=0)

        total_importance = self.feature_importances_.sum()
        if total_importance > 0:
            self.feature_importances_ /= total_importance
        return self

    def _make_leaf(self, indices):
        n = len(indices)
        positive = float(self.y_[indices].sum()) if n else 0.0
        negative = float(n) - positive
        weighted_positive = self.positive_class_weight * positive
        denom = weighted_positive + negative
        probability = weighted_positive / denom if denom > 0 else 0.0
        return {
            "is_leaf": True,
            "probability": float(probability),
            "prediction": int(probability >= 0.5),
            "n_samples": int(n),
        }

    def _get_feature_subset(self):
        if self.max_features is None or self.max_features >= self.n_features_in_:
            return np.arange(self.n_features_in_)
        if self.max_features == "sqrt":
            n_features = max(1, int(np.sqrt(self.n_features_in_)))
        else:
            n_features = int(self.max_features)
        return self.rng_.choice(
            self.n_features_in_, size=n_features, replace=False
        )

    def _best_split(self, indices):
        y_node = self.y_[indices]
        n_node = len(indices)
        total_positive = float(y_node.sum())
        total_negative = float(n_node) - total_positive

        weighted_positive_parent = self.positive_class_weight * total_positive
        weighted_negative_parent = total_negative
        parent_weight = weighted_positive_parent + weighted_negative_parent
        parent_gini = self._weighted_gini(total_positive, n_node)

        best_gain = 0.0
        best_split = None

        for feature_index in self._get_feature_subset():
            values = self.X_[indices, feature_index]
            order = np.argsort(values, kind="mergesort")
            sorted_values = values[order]
            sorted_y = y_node[order]

            candidate_positions = np.flatnonzero(
                sorted_values[:-1] < sorted_values[1:]
            )
            candidate_positions = candidate_positions[
                (candidate_positions + 1 >= self.min_samples_leaf)
                & (n_node - (candidate_positions + 1) >= self.min_samples_leaf)
            ]
            if len(candidate_positions) == 0:
                continue

            if self.max_thresholds is not None and len(candidate_positions) > self.max_thresholds:
                selected = np.linspace(
                    0, len(candidate_positions) - 1,
                    self.max_thresholds, dtype=int
                )
                candidate_positions = candidate_positions[selected]

            cumulative_positive = np.cumsum(sorted_y)
            n_left = (candidate_positions + 1).astype(float)
            n_right = float(n_node) - n_left

            positive_left = cumulative_positive[candidate_positions].astype(float)
            positive_right = total_positive - positive_left
            negative_left = n_left - positive_left
            negative_right = n_right - positive_right

            weighted_positive_left = self.positive_class_weight * positive_left
            weighted_positive_right = self.positive_class_weight * positive_right
            weight_left = weighted_positive_left + negative_left
            weight_right = weighted_positive_right + negative_right

            p_left = weighted_positive_left / weight_left
            p_right = weighted_positive_right / weight_right
            gini_left = 2.0 * p_left * (1.0 - p_left)
            gini_right = 2.0 * p_right * (1.0 - p_right)

            gains = parent_gini - (
                (weight_left / parent_weight) * gini_left
                + (weight_right / parent_weight) * gini_right
            )

            local_best = int(np.argmax(gains))
            gain = float(gains[local_best])
            if gain <= best_gain + 1e-12:
                continue

            split_position = int(candidate_positions[local_best])
            threshold = float(
                (sorted_values[split_position] + sorted_values[split_position + 1]) / 2.0
            )
            left_mask = values <= threshold

            best_gain = gain
            best_split = (
                feature_index,
                threshold,
                indices[left_mask],
                indices[~left_mask],
                gain,
            )

        return best_split

    def _grow(self, indices, depth):
        depth_stop = self.max_depth is not None and depth >= self.max_depth
        stopping_condition = (
            depth_stop
            or len(indices) < self.min_samples_split
            or np.unique(self.y_[indices]).size == 1
        )
        if stopping_condition:
            return self._make_leaf(indices)

        split = self._best_split(indices)
        if split is None:
            return self._make_leaf(indices)

        feature_index, threshold, left_indices, right_indices, gain = split
        positive = float(self.y_[indices].sum())
        node_weight = self.positive_class_weight * positive + (len(indices) - positive)
        self.feature_importances_[feature_index] += gain * node_weight

        return {
            "is_leaf": False,
            "feature_index": int(feature_index),
            "threshold": float(threshold),
            "n_samples": int(len(indices)),
            "left": self._grow(left_indices, depth + 1),
            "right": self._grow(right_indices, depth + 1),
        }

    def _predict_probability_one(self, row):
        node = self.tree_
        while not node["is_leaf"]:
            if row[node["feature_index"]] <= node["threshold"]:
                node = node["left"]
            else:
                node = node["right"]
        return node["probability"]

    def predict_proba(self, X):
        X = np.asarray(X, dtype=float)
        positive_probability = np.fromiter(
            (self._predict_probability_one(row) for row in X),
            dtype=float,
            count=len(X),
        )
        return np.column_stack([1.0 - positive_probability, positive_probability])

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X)[:, 1] >= threshold).astype(int)


## 2. Logistic Regression

Probabilitas kelas positif diperoleh melalui fungsi sigmoid:

\[
\hat{p}=\sigma(Xw+b)=\frac{1}{1+e^{-(Xw+b)}}
\]

Parameter dioptimalkan dengan weighted binary cross-entropy dan regularisasi L2. Bobot kelas dihitung manual agar kelas minoritas tidak diabaikan. Optimizer Adam juga ditulis langsung menggunakan NumPy.


In [ ]:
class LogisticRegressionScratch:
    def __init__(
        self,
        learning_rate=0.03,
        epochs=250,
        batch_size=512,
        l2=0.001,
        class_weight="balanced",
        tolerance=1e-6,
        patience=20,
        random_state=42,
    ):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        self.l2 = l2
        self.class_weight = class_weight
        self.tolerance = tolerance
        self.patience = patience
        self.random_state = random_state

    @staticmethod
    def _sigmoid(z):
        z = np.clip(z, -35, 35)
        return 1.0 / (1.0 + np.exp(-z))

    def _sample_weights(self, y):
        if self.class_weight != "balanced":
            return np.ones(len(y), dtype=float)
        n_samples = len(y)
        n_negative = max(int((y == 0).sum()), 1)
        n_positive = max(int((y == 1).sum()), 1)
        weight_negative = n_samples / (2.0 * n_negative)
        weight_positive = n_samples / (2.0 * n_positive)
        return np.where(y == 1, weight_positive, weight_negative)

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        n_samples, n_features = X.shape
        rng = np.random.default_rng(self.random_state)

        self.coef_ = np.zeros(n_features, dtype=float)
        self.intercept_ = 0.0
        sample_weights = self._sample_weights(y)

        # State Adam optimizer
        m_w = np.zeros(n_features)
        v_w = np.zeros(n_features)
        m_b = 0.0
        v_b = 0.0
        beta1, beta2, eps = 0.9, 0.999, 1e-8
        step = 0

        self.loss_history_ = []
        best_loss = np.inf
        stale_epochs = 0

        for epoch in range(self.epochs):
            order = rng.permutation(n_samples)

            for start in range(0, n_samples, self.batch_size):
                idx = order[start:start + self.batch_size]
                X_batch = X[idx]
                y_batch = y[idx]
                weight_batch = sample_weights[idx]

                probability = self._sigmoid(
                    X_batch @ self.coef_ + self.intercept_
                )
                weighted_error = (probability - y_batch) * weight_batch

                grad_w = (
                    X_batch.T @ weighted_error / len(idx)
                    + self.l2 * self.coef_
                )
                grad_b = weighted_error.mean()

                step += 1
                m_w = beta1 * m_w + (1 - beta1) * grad_w
                v_w = beta2 * v_w + (1 - beta2) * (grad_w ** 2)
                m_b = beta1 * m_b + (1 - beta1) * grad_b
                v_b = beta2 * v_b + (1 - beta2) * (grad_b ** 2)

                m_w_hat = m_w / (1 - beta1 ** step)
                v_w_hat = v_w / (1 - beta2 ** step)
                m_b_hat = m_b / (1 - beta1 ** step)
                v_b_hat = v_b / (1 - beta2 ** step)

                self.coef_ -= self.learning_rate * m_w_hat / (np.sqrt(v_w_hat) + eps)
                self.intercept_ -= self.learning_rate * m_b_hat / (np.sqrt(v_b_hat) + eps)

            probability_full = self._sigmoid(X @ self.coef_ + self.intercept_)
            log_loss = -np.mean(
                sample_weights * (
                    y * np.log(probability_full + 1e-12)
                    + (1 - y) * np.log(1 - probability_full + 1e-12)
                )
            )
            loss = log_loss + 0.5 * self.l2 * np.dot(self.coef_, self.coef_)
            self.loss_history_.append(float(loss))

            if best_loss - loss > self.tolerance:
                best_loss = loss
                stale_epochs = 0
            else:
                stale_epochs += 1

            if stale_epochs >= self.patience:
                break

        self.n_iter_ = len(self.loss_history_)
        return self

    def predict_proba(self, X):
        positive_probability = self._sigmoid(
            np.asarray(X, dtype=float) @ self.coef_ + self.intercept_
        )
        return np.column_stack([1.0 - positive_probability, positive_probability])

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X)[:, 1] >= threshold).astype(int)


## 3. Linear Support Vector Machine

Label diubah menjadi \(-1\) dan \(+1\). SVM mencari hyperplane dengan margin besar melalui objective:

\[
\frac{\lambda}{2}\lVert w\rVert^2+
\frac{1}{n}\sum_i \max(0,1-y_i(w^Tx_i+b))
\]

Bagian \(\max(0,1-margin)\) disebut **hinge loss**. Gradien hanya menerima kontribusi dari observasi yang marginnya kurang dari 1.


In [ ]:
class LinearSVMScratch:
    def __init__(
        self,
        learning_rate=0.01,
        epochs=250,
        batch_size=512,
        regularization=0.001,
        class_weight="balanced",
        random_state=42,
    ):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.batch_size = batch_size
        self.regularization = regularization
        self.class_weight = class_weight
        self.random_state = random_state

    def _sample_weights(self, y_signed):
        if self.class_weight != "balanced":
            return np.ones(len(y_signed), dtype=float)
        n_samples = len(y_signed)
        n_negative = max(int((y_signed == -1).sum()), 1)
        n_positive = max(int((y_signed == 1).sum()), 1)
        weight_negative = n_samples / (2.0 * n_negative)
        weight_positive = n_samples / (2.0 * n_positive)
        return np.where(y_signed == 1, weight_positive, weight_negative)

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y_signed = np.where(np.asarray(y) == 1, 1.0, -1.0)
        n_samples, n_features = X.shape
        rng = np.random.default_rng(self.random_state)

        self.coef_ = np.zeros(n_features, dtype=float)
        self.intercept_ = 0.0
        sample_weights = self._sample_weights(y_signed)
        self.loss_history_ = []

        for epoch in range(self.epochs):
            order = rng.permutation(n_samples)
            current_lr = self.learning_rate / (1.0 + 0.02 * epoch)

            for start in range(0, n_samples, self.batch_size):
                idx = order[start:start + self.batch_size]
                X_batch = X[idx]
                y_batch = y_signed[idx]
                weight_batch = sample_weights[idx]

                margins = y_batch * (X_batch @ self.coef_ + self.intercept_)
                active = margins < 1.0

                grad_w = self.regularization * self.coef_
                grad_b = 0.0

                if np.any(active):
                    active_coefficient = weight_batch[active] * y_batch[active]
                    grad_w -= (
                        X_batch[active].T @ active_coefficient / len(idx)
                    )
                    grad_b -= active_coefficient.sum() / len(idx)

                self.coef_ -= current_lr * grad_w
                self.intercept_ -= current_lr * grad_b

            full_margin = y_signed * (X @ self.coef_ + self.intercept_)
            hinge = np.maximum(0.0, 1.0 - full_margin)
            loss = (
                0.5 * self.regularization * np.dot(self.coef_, self.coef_)
                + np.mean(sample_weights * hinge)
            )
            self.loss_history_.append(float(loss))

        return self

    def decision_function(self, X):
        return np.asarray(X, dtype=float) @ self.coef_ + self.intercept_

    def predict(self, X):
        return (self.decision_function(X) >= 0.0).astype(int)


## Training Ketiga Model From Scratch


In [ ]:
scratch_model_factories = {
    "CART Scratch": lambda: CARTClassifierScratch(
        max_depth=7,
        min_samples_split=30,
        min_samples_leaf=15,
        max_thresholds=48,
        random_state=SEED,
    ),
    "Logistic Regression Scratch": lambda: LogisticRegressionScratch(
        learning_rate=0.03,
        epochs=250,
        batch_size=512,
        l2=0.001,
        class_weight="balanced",
        random_state=SEED,
    ),
    "Linear SVM Scratch": lambda: LinearSVMScratch(
        learning_rate=0.01,
        epochs=250,
        batch_size=512,
        regularization=0.001,
        class_weight="balanced",
        random_state=SEED,
    ),
}

scratch_models = {}
scratch_train_times = {}

for model_name, factory in scratch_model_factories.items():
    model = factory()
    start_time = time.perf_counter()
    model.fit(X_train, y_train)
    elapsed = time.perf_counter() - start_time
    scratch_models[model_name] = model
    scratch_train_times[model_name] = elapsed
    print(f"{model_name:30s} selesai dalam {elapsed:.4f} detik")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(scratch_models["Logistic Regression Scratch"].loss_history_)
axes[0].set_title("Training Loss — Logistic Regression Scratch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Weighted BCE + L2")

axes[1].plot(scratch_models["Linear SVM Scratch"].loss_history_)
axes[1].set_title("Training Loss — Linear SVM Scratch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Hinge Loss + L2")

plt.tight_layout()
plt.show()


<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Pembanding Scikit-learn <a name="8"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

Pembanding menggunakan keluarga algoritma yang sama dan data preprocessing yang identik:

- `DecisionTreeClassifier` untuk CART;
- `LogisticRegression` untuk logistic regression;
- `LinearSVC` untuk linear SVM.


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

sklearn_model_factories = {
    "CART Scikit-learn": lambda: DecisionTreeClassifier(
        criterion="gini",
        max_depth=7,
        min_samples_split=30,
        min_samples_leaf=15,
        random_state=SEED,
    ),
    "Logistic Regression Scikit-learn": lambda: LogisticRegression(
        class_weight="balanced",
        max_iter=3000,
        random_state=SEED,
    ),
    "Linear SVM Scikit-learn": lambda: LinearSVC(
        C=1.0,
        class_weight="balanced",
        max_iter=10000,
        random_state=SEED,
    ),
}

sklearn_models = {}
sklearn_train_times = {}

for model_name, factory in sklearn_model_factories.items():
    model = factory()
    start_time = time.perf_counter()
    model.fit(X_train, y_train)
    elapsed = time.perf_counter() - start_time
    sklearn_models[model_name] = model
    sklearn_train_times[model_name] = elapsed
    print(f"{model_name:35s} selesai dalam {elapsed:.4f} detik")


<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Evaluasi Baseline <a name="9"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

Bagian ini mereproduksi perbandingan awal pada satu holdout. Hasilnya berguna sebagai baseline, tetapi **bukan satu-satunya dasar pemilihan submission final**.


## Fungsi Evaluasi


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
)


def get_model_score(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    return model.decision_function(X)


def evaluate_model(model_name, implementation, model, X, y, train_time):
    
    prediction = model.predict(X)
    score = get_model_score(model, X)

    return {
        "model": model_name,
        "implementation": implementation,
        "accuracy": accuracy_score(y, prediction),
        "precision_class_1": precision_score(
            y, prediction, pos_label=1, zero_division=0
        ),
        "recall_class_1": recall_score(
            y, prediction, pos_label=1, zero_division=0
        ),
        "f1_class_0": f1_score(
            y, prediction, pos_label=0, zero_division=0
        ),
        "f1_class_1": f1_score(
            y, prediction, pos_label=1, zero_division=0
        ),
        # Metrik resmi kompetisi: rata-rata F1 dari kelas 0 dan kelas 1.
        "macro_f1": f1_score(
            y, prediction, average="macro", zero_division=0
        ),
        "roc_auc": roc_auc_score(y, score),
        "train_time_seconds": train_time,
    }


In [ ]:
evaluation_rows = []

for name, model in scratch_models.items():
    evaluation_rows.append(
        evaluate_model(
            name, "From Scratch", model,
            X_valid, y_valid, scratch_train_times[name]
        )
    )

for name, model in sklearn_models.items():
    evaluation_rows.append(
        evaluate_model(
            name, "Scikit-learn", model,
            X_valid, y_valid, sklearn_train_times[name]
        )
    )

# Ranking utama mengikuti metrik resmi kompetisi: macro F1.
results_df = pd.DataFrame(evaluation_rows).sort_values(
    ["macro_f1", "roc_auc"], ascending=False
).reset_index(drop=True)

results_df.style.format({
    "accuracy": "{:.4f}",
    "precision_class_1": "{:.4f}",
    "recall_class_1": "{:.4f}",
    "f1_class_0": "{:.4f}",
    "f1_class_1": "{:.4f}",
    "macro_f1": "{:.4f}",
    "roc_auc": "{:.4f}",
    "train_time_seconds": "{:.4f}",
}).background_gradient(
    subset=["f1_class_0", "f1_class_1", "macro_f1"],
    cmap="YlGn"
)


## Perbandingan Metrik


In [ ]:
metric_columns = [
    "accuracy", "f1_class_0", "f1_class_1", "macro_f1", "roc_auc"
]
plot_data = results_df.set_index("model")[metric_columns]

ax = plot_data.plot(kind="bar", figsize=(15, 6))
ax.set_ylim(0.5, 1.0)
ax.set_title("Perbandingan Model — Macro F1 sebagai Metrik Utama")
ax.set_ylabel("Score")
ax.set_xlabel("")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


## Confusion Matrix — Semua Model


In [ ]:
all_models = {**scratch_models, **sklearn_models}
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.reshape(-1)

for ax, (name, model) in zip(axes, all_models.items()):
    cm = confusion_matrix(y_valid, model.predict(X_valid))
    image = ax.imshow(cm)
    ax.set_title(name, fontsize=10)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center")

plt.tight_layout()
plt.show()


## ROC Curve


In [ ]:
plt.figure(figsize=(9, 6))
for name, model in all_models.items():
    score = get_model_score(model, X_valid)
    fpr, tpr, _ = roc_curve(y_valid, score)
    auc_value = roc_auc_score(y_valid, score)
    plt.plot(fpr, tpr, label=f"{name} ({auc_value:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend(fontsize=8)
plt.grid(alpha=0.25)
plt.show()


## Feature Importance CART Scratch


In [ ]:
cart_scratch = scratch_models["CART Scratch"]
importance_df = pd.DataFrame({
    "feature": preprocessor.feature_names_,
    "importance": cart_scratch.feature_importances_,
}).sort_values("importance", ascending=False)

display(importance_df.head(12))

ax = importance_df.head(12).sort_values("importance").plot(
    kind="barh", x="feature", y="importance", figsize=(9, 6), legend=False
)
ax.set_title("Top Feature Importance — CART Scratch")
ax.set_xlabel("Normalized Gini Gain")
plt.tight_layout()
plt.show()


## Classification Report Model Scratch Terbaik


In [ ]:
scratch_results = results_df[
    results_df["implementation"] == "From Scratch"
].copy()
best_scratch_name = scratch_results.iloc[0]["model"]
best_scratch_model = scratch_models[best_scratch_name]

print("Model scratch terbaik berdasarkan macro F1:", best_scratch_name)
print(
    "Macro F1 validation:",
    f"{scratch_results.iloc[0]['macro_f1']:.5f}"
)
print()
print(classification_report(
    y_valid,
    best_scratch_model.predict(X_valid),
    digits=4,
))


**Interpretasi evaluasi**

- Kompetisi memakai **macro F1-score**, yaitu rata-rata F1 kelas `0` dan F1 kelas `1`. Karena kedua kelas diberi bobot yang sama, performa pada kelas mayoritas tidak dapat menutupi performa yang buruk pada kelas minoritas.
- Model scratch terbaik dan file submission utama dipilih berdasarkan `macro_f1`, bukan accuracy maupun F1 kelas `1` saja.
- `f1_class_0` dan `f1_class_1` tetap ditampilkan agar keseimbangan performa antar-kelas dapat diperiksa.
- ROC-AUC menilai kualitas ranking score di seluruh threshold, sedangkan confusion matrix menunjukkan false positive dan false negative secara langsung.
- Perbedaan hasil scratch dan scikit-learn dapat muncul karena optimisasi, strategi threshold, stopping criterion, regularisasi, dan detail implementasi numerik.


<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Robust Model Development V2 — Weighted CART, No Leakage <a name="10"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

Target kompetisi adalah **macro F1**, sementara kelas positif lebih sedikit. Pada V1, CART memakai impurity biasa lalu hanya menyesuaikan decision threshold. V2 menambahkan **cost-sensitive CART**: bobot kelas positif ikut masuk ke perhitungan Gini ketika tree memilih split.

Perbaikan utama:

1. **Class-weighted Gini pada CART scratch** — positive class weight dituning dari train OOF saja.
2. **Unbounded depth + regularisasi minimum leaf** — tree boleh tumbuh selama split masih valid, tetapi `min_samples_leaf` mencegah leaf terlalu kecil.
3. **Exact split search** — seluruh threshold numerik yang valid diperiksa.
4. **5-fold OOF tuning + repeated-seed stability audit**.
5. **Exact macro-F1 threshold optimization** pada prediksi OOF.
6. Logistic Regression dan Linear SVM tetap dipertahankan sebagai implementasi scratch dan pembanding yang diwajibkan.

> Class weighting bukan ensemble dan bukan algoritma tambahan. Submission tetap berasal dari **satu CART from scratch**. `person_id` tetap tidak digunakan sebagai feature.


## Utilitas Stratified K-Fold dan Exact Threshold Optimization

Macro F1 sensitif terhadap threshold. Threshold `0.5` pada Logistic Regression atau `0.0` pada SVM tidak selalu optimal, terutama saat kelas tidak seimbang. Fungsi berikut mencari threshold terbaik dari OOF score secara `O(n log n)` melalui sorting dan cumulative confusion counts.


In [ ]:
def stratified_kfold_indices(y, n_splits=5, random_state=42):
    
    y = np.asarray(y)
    rng = np.random.default_rng(random_state)
    fold_validation = [[] for _ in range(n_splits)]

    for cls in np.unique(y):
        cls_idx = np.flatnonzero(y == cls)
        rng.shuffle(cls_idx)
        for fold_id, part in enumerate(np.array_split(cls_idx, n_splits)):
            fold_validation[fold_id].extend(part.tolist())

    all_idx = np.arange(len(y))
    for fold_id in range(n_splits):
        valid_idx_fold = np.asarray(fold_validation[fold_id], dtype=int)
        train_mask = np.ones(len(y), dtype=bool)
        train_mask[valid_idx_fold] = False
        train_idx_fold = all_idx[train_mask]
        yield train_idx_fold, valid_idx_fold


def macro_f1_numpy(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)

    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    tn = np.sum((y_true == 0) & (y_pred == 0))

    f1_pos = 2.0 * tp / max(2 * tp + fp + fn, 1)
    f1_neg = 2.0 * tn / max(2 * tn + fp + fn, 1)
    return 0.5 * (f1_pos + f1_neg)


def optimize_macro_f1_threshold(y_true, score):
    
    y_true = np.asarray(y_true, dtype=int)
    score = np.asarray(score, dtype=float)

    order = np.argsort(-score, kind="mergesort")
    y_sorted = y_true[order]
    s_sorted = score[order]

    # k = jumlah observasi yang diprediksi positif, termasuk k=0.
    tp = np.r_[0, np.cumsum(y_sorted == 1)]
    fp = np.r_[0, np.cumsum(y_sorted == 0)]
    total_pos = int((y_true == 1).sum())
    total_neg = int((y_true == 0).sum())
    fn = total_pos - tp
    tn = total_neg - fp

    f1_pos = 2.0 * tp / np.maximum(2.0 * tp + fp + fn, 1.0)
    f1_neg = 2.0 * tn / np.maximum(2.0 * tn + fp + fn, 1.0)
    macro = 0.5 * (f1_pos + f1_neg)

    # Threshold hanya valid saat berpindah ke nilai score yang berbeda.
    valid_k = [0]
    if len(score) > 1:
        valid_k.extend((np.flatnonzero(s_sorted[:-1] > s_sorted[1:]) + 1).tolist())
    valid_k.append(len(score))
    valid_k = np.asarray(sorted(set(valid_k)), dtype=int)

    best_k = int(valid_k[np.argmax(macro[valid_k])])
    if best_k == 0:
        threshold = float(s_sorted[0] + 1e-12)
    elif best_k == len(score):
        threshold = float(s_sorted[-1] - 1e-12)
    else:
        threshold = float((s_sorted[best_k - 1] + s_sorted[best_k]) / 2.0)

    return threshold, float(macro[best_k])


## Nonlinear Feature Expansion untuk Logistic Regression dan SVM

CART sudah bersifat nonlinear sehingga menggunakan fitur dasar. Untuk model linear, ditambahkan basis threshold berbasis quantile dan interaksi dengan `person_home_ownership`. Transformasi ini **bukan model baru**: prediktor akhirnya tetap Logistic Regression atau Linear SVM. Cut-point quantile dipelajari tanpa melihat `y` dan selalu di-fit ulang pada training fold.


In [ ]:
class ScratchNonlinearPreprocessor:
    
    def __init__(self, n_quantiles=15):
        self.n_quantiles = n_quantiles

    def fit(self, frame):
        self.numeric_cols_ = [
            c for c in frame.columns if pd.api.types.is_numeric_dtype(frame[c])
        ]
        self.categorical_cols_ = [
            c for c in frame.columns if c not in self.numeric_cols_
        ]

        self.medians_ = {
            c: float(frame[c].median()) for c in self.numeric_cols_
        }
        numeric = np.column_stack([
            frame[c].fillna(self.medians_[c]).to_numpy(dtype=float)
            for c in self.numeric_cols_
        ])
        self.means_ = numeric.mean(axis=0)
        self.stds_ = numeric.std(axis=0)
        self.stds_[self.stds_ < 1e-12] = 1.0

        self.categories_ = {
            c: sorted(frame[c].fillna("__MISSING__").astype(str).unique().tolist())
            for c in self.categorical_cols_
        }

        candidate_keys = [
            "loan_percent_income",
            "loan_int_rate",
            "person_income",
            "credit_score",
            "loan_amnt",
        ]
        self.threshold_cols_ = [c for c in candidate_keys if c in frame.columns]
        quantile_grid = np.linspace(0.05, 0.95, self.n_quantiles)
        self.quantile_thresholds_ = {}
        for c in self.threshold_cols_:
            values = frame[c].fillna(self.medians_[c]).to_numpy(dtype=float)
            self.quantile_thresholds_[c] = np.unique(
                np.quantile(values, quantile_grid)
            )

        self.feature_names_ = list(self.numeric_cols_)
        for c in self.categorical_cols_:
            self.feature_names_.extend([
                f"{c}={category}" for category in self.categories_[c]
            ])
        for c in self.threshold_cols_:
            self.feature_names_.extend([
                f"{c}>q{i:02d}" for i in range(len(self.quantile_thresholds_[c]))
            ])

        if (
            "person_home_ownership" in self.categories_
            and "previous_loan_defaults_on_file" in self.categories_
        ):
            self.feature_names_.extend([
                f"home={h}|prev_default={d}"
                for h in self.categories_["person_home_ownership"]
                for d in self.categories_["previous_loan_defaults_on_file"]
            ])

        for c in ["loan_percent_income", "loan_int_rate", "person_income"]:
            if c in self.quantile_thresholds_ and "person_home_ownership" in self.categories_:
                self.feature_names_.extend([
                    f"{c}>q{i:02d}|home={h}"
                    for i in range(len(self.quantile_thresholds_[c]))
                    for h in self.categories_["person_home_ownership"]
                ])
        return self

    def transform(self, frame):
        numeric = np.column_stack([
            frame[c].fillna(self.medians_[c]).to_numpy(dtype=float)
            for c in self.numeric_cols_
        ])
        blocks = [(numeric - self.means_) / self.stds_]

        categorical_blocks = {}
        for c in self.categorical_cols_:
            values = frame[c].fillna("__MISSING__").astype(str).to_numpy()
            block = np.column_stack([
                (values == category).astype(float)
                for category in self.categories_[c]
            ])
            categorical_blocks[c] = block
            blocks.append(block)

        threshold_blocks = {}
        for c in self.threshold_cols_:
            values = frame[c].fillna(self.medians_[c]).to_numpy(dtype=float)
            block = np.column_stack([
                (values > threshold).astype(float)
                for threshold in self.quantile_thresholds_[c]
            ])
            threshold_blocks[c] = block
            blocks.append(block)

        if (
            "person_home_ownership" in categorical_blocks
            and "previous_loan_defaults_on_file" in categorical_blocks
        ):
            home = categorical_blocks["person_home_ownership"]
            previous = categorical_blocks["previous_loan_defaults_on_file"]
            blocks.append(
                (home[:, :, None] * previous[:, None, :]).reshape(len(frame), -1)
            )

        if "person_home_ownership" in categorical_blocks:
            home = categorical_blocks["person_home_ownership"]
            for c in ["loan_percent_income", "loan_int_rate", "person_income"]:
                if c in threshold_blocks:
                    threshold_block = threshold_blocks[c]
                    blocks.append(
                        (threshold_block[:, :, None] * home[:, None, :])
                        .reshape(len(frame), -1)
                    )

        return np.column_stack(blocks).astype(np.float64)

    def fit_transform(self, frame):
        return self.fit(frame).transform(frame)


## Hyperparameter Search V2 — Fokus pada Cost-Sensitive CART

Grid V2 menambahkan `positive_class_weight` pada CART. Bobot ini hanya mengubah kontribusi kelas positif di **Gini impurity** dan estimasi probabilitas leaf; bukan resampling, boosting, bagging, atau ensemble.

Semua kandidat dipilih dari **train OOF**. `test.csv` dan public leaderboard tidak dipakai untuk memilih parameter atau threshold.


In [ ]:
ROBUST_TUNING_GRIDS = {
    "CART Scratch": [
        {"max_depth": 8, "min_samples_leaf": 40, "positive_class_weight": 1.0},
        {"max_depth": None, "min_samples_leaf": 30, "positive_class_weight": 1.0},
        {"max_depth": None, "min_samples_leaf": 40, "positive_class_weight": 1.0},
        {"max_depth": None, "min_samples_leaf": 40, "positive_class_weight": 1.1},
        {"max_depth": None, "min_samples_leaf": 40, "positive_class_weight": 1.2},
        {"max_depth": None, "min_samples_leaf": 40, "positive_class_weight": 1.3},
        {"max_depth": None, "min_samples_leaf": 50, "positive_class_weight": 1.3},
    ],
    "Logistic Regression Scratch": [
        {"l2": 0.001},
        {"l2": 0.003},
    ],
    "Linear SVM Scratch": [
        {"regularization": 0.001},
        {"regularization": 0.003},
    ],
}


def make_robust_preprocessor(model_name):
    if model_name == "CART Scratch":
        return ScratchTabularPreprocessor()
    if model_name == "Logistic Regression Scratch":
        return ScratchNonlinearPreprocessor(n_quantiles=15)
    return ScratchNonlinearPreprocessor(n_quantiles=9)


def make_robust_model(model_name, params, seed=SEED):
    if model_name == "CART Scratch":
        leaf = int(params["min_samples_leaf"])
        depth = params.get("max_depth", None)
        return CARTClassifierScratch(
            max_depth=None if depth is None else int(depth),
            min_samples_split=2 * leaf,
            min_samples_leaf=leaf,
            max_thresholds=None,
            positive_class_weight=float(params.get("positive_class_weight", 1.0)),
            random_state=seed,
        )

    if model_name == "Logistic Regression Scratch":
        return LogisticRegressionScratch(
            learning_rate=0.02,
            epochs=180,
            batch_size=512,
            l2=float(params["l2"]),
            class_weight=None,
            patience=25,
            random_state=seed,
        )

    return LinearSVMScratch(
        learning_rate=0.01,
        epochs=60,
        batch_size=512,
        regularization=float(params["regularization"]),
        class_weight=None,
        random_state=seed,
    )


def run_oof_cv(model_name, params, cv_seed=2026, n_splits=5):
    y_all = train_df[TARGET].to_numpy(dtype=int)
    oof_score = np.zeros(len(train_df), dtype=float)
    fold_rows = []
    start_time = time.perf_counter()

    for fold_id, (tr_idx, va_idx) in enumerate(
        stratified_kfold_indices(y_all, n_splits=n_splits, random_state=cv_seed),
        start=1,
    ):
        pre = make_robust_preprocessor(model_name)
        X_tr = pre.fit_transform(train_df.iloc[tr_idx][feature_cols])
        X_va = pre.transform(train_df.iloc[va_idx][feature_cols])

        model = make_robust_model(model_name, params, seed=SEED + fold_id)
        model.fit(X_tr, y_all[tr_idx])
        score = get_model_score(model, X_va)
        oof_score[va_idx] = score

        default_threshold = 0.0 if model_name == "Linear SVM Scratch" else 0.5
        fold_rows.append({
            "fold": fold_id,
            "macro_f1_default": macro_f1_numpy(
                y_all[va_idx], (score >= default_threshold).astype(int)
            ),
        })

    threshold, macro_f1_oof = optimize_macro_f1_threshold(y_all, oof_score)
    elapsed = time.perf_counter() - start_time

    return {
        "model": model_name,
        "params": params.copy(),
        "cv_seed": cv_seed,
        "threshold": threshold,
        "macro_f1_oof": macro_f1_oof,
        "roc_auc_oof": roc_auc_score(y_all, oof_score),
        "time_seconds": elapsed,
        "oof_score": oof_score,
        "fold_rows": fold_rows,
    }


In [ ]:
# Phase A — compact 5-fold tuning pada satu seed.
TUNING_SEED = 2026
tuning_results = []
robust_cv_cache = {}

for model_name, grid in ROBUST_TUNING_GRIDS.items():
    for params in grid:
        result = run_oof_cv(
            model_name=model_name,
            params=params,
            cv_seed=TUNING_SEED,
            n_splits=5,
        )
        tuning_results.append({
            "model": model_name,
            "params": str(params),
            "threshold": result["threshold"],
            "macro_f1_oof": result["macro_f1_oof"],
            "roc_auc_oof": result["roc_auc_oof"],
            "time_seconds": result["time_seconds"],
        })
        robust_cv_cache[(model_name, str(params), TUNING_SEED)] = result
        print(
            f"{model_name:30s} {params} -> "
            f"OOF macro F1={result['macro_f1_oof']:.5f} | "
            f"threshold={result['threshold']:.5f}"
        )

robust_tuning_df = pd.DataFrame(tuning_results).sort_values(
    ["model", "macro_f1_oof"], ascending=[True, False]
).reset_index(drop=True)

display(robust_tuning_df)


## Stability Audit — Repeated 5-Fold CV pada Dua Kandidat Teratas

Dua konfigurasi dengan OOF macro F1 tertinggi dari Phase A diuji ulang pada tiga seed 5-fold. Kandidat final dipilih berdasarkan **mean repeated OOF macro F1**, lalu threshold final diambil dari median threshold antar-repeat. Repeated CV hanya dipakai untuk evaluasi/tuning; submission tetap berasal dari satu model manual.


In [ ]:
best_params_by_model = {}
best_threshold_by_model = {}
best_oof_by_model = {}

for model_name in ROBUST_TUNING_GRIDS:
    family_rows = robust_tuning_df[
        robust_tuning_df["model"] == model_name
    ].sort_values("macro_f1_oof", ascending=False)
    best_row = family_rows.iloc[0]
    best_params_text = best_row["params"]
    best_params_by_model[model_name] = next(
        p for p in ROBUST_TUNING_GRIDS[model_name]
        if str(p) == best_params_text
    )
    best_threshold_by_model[model_name] = float(best_row["threshold"])
    best_oof_by_model[model_name] = float(best_row["macro_f1_oof"])

family_best_df = pd.DataFrame([
    {
        "model": model_name,
        "params": str(best_params_by_model[model_name]),
        "macro_f1_oof": best_oof_by_model[model_name],
        "threshold": best_threshold_by_model[model_name],
    }
    for model_name in ROBUST_TUNING_GRIDS
]).sort_values("macro_f1_oof", ascending=False).reset_index(drop=True)

display(family_best_df)

# Tiga konfigurasi terbaik secara global masuk stability audit.
finalist_df = robust_tuning_df.sort_values(
    ["macro_f1_oof", "roc_auc_oof"], ascending=False
).head(3).reset_index(drop=True)

STABILITY_SEEDS = [42, 2026, 777]
stability_rows = []

for finalist_id, row in finalist_df.iterrows():
    model_name = row["model"]
    params_text = row["params"]
    params = next(
        p for p in ROBUST_TUNING_GRIDS[model_name]
        if str(p) == params_text
    )

    for cv_seed in STABILITY_SEEDS:
        cache_key = (model_name, str(params), cv_seed)
        result = robust_cv_cache.get(cache_key)
        if result is None:
            result = run_oof_cv(
                model_name=model_name,
                params=params,
                cv_seed=cv_seed,
                n_splits=5,
            )
        stability_rows.append({
            "candidate_id": finalist_id,
            "model": model_name,
            "params": str(params),
            "cv_seed": cv_seed,
            "threshold": result["threshold"],
            "macro_f1_oof": result["macro_f1_oof"],
            "roc_auc_oof": result["roc_auc_oof"],
        })

stability_df = pd.DataFrame(stability_rows)
display(stability_df.sort_values(["candidate_id", "cv_seed"]))

stability_summary = (
    stability_df.groupby(["candidate_id", "model", "params"], as_index=False)
    .agg(
        mean_macro_f1=("macro_f1_oof", "mean"),
        std_macro_f1=("macro_f1_oof", "std"),
        mean_roc_auc=("roc_auc_oof", "mean"),
        median_threshold=("threshold", "median"),
    )
    .sort_values(["mean_macro_f1", "mean_roc_auc"], ascending=False)
    .reset_index(drop=True)
)

display(stability_summary.style.format({
    "mean_macro_f1": "{:.5f}",
    "std_macro_f1": "{:.5f}",
    "mean_roc_auc": "{:.5f}",
    "median_threshold": "{:.5f}",
}))

best_stability_row = stability_summary.iloc[0]
best_robust_model_name = best_stability_row["model"]
best_params_text = best_stability_row["params"]
best_robust_params = next(
    p for p in ROBUST_TUNING_GRIDS[best_robust_model_name]
    if str(p) == best_params_text
)
best_robust_threshold = float(best_stability_row["median_threshold"])

# Threshold model non-final memakai hasil best 5-fold Phase A; kandidat final
 # memakai median repeated-CV agar lebih stabil.
threshold_by_model = best_threshold_by_model.copy()
threshold_by_model[best_robust_model_name] = best_robust_threshold
best_params_by_model[best_robust_model_name] = best_robust_params

print("Model robust terpilih  :", best_robust_model_name)
print("Hyperparameter         :", best_robust_params)
print("Median OOF threshold   :", f"{best_robust_threshold:.6f}")
print("Mean repeated OOF F1   :", f"{best_stability_row['mean_macro_f1']:.6f}")
print("Std repeated OOF F1    :", f"{best_stability_row['std_macro_f1']:.6f}")


### Mengapa V2 Lebih Robust?

- `person_id` **tetap tidak digunakan** sebagai feature walaupun identifier dapat mengandung pola artifisial pada dataset.
- Class weighting dipelajari sebagai hyperparameter dari train OOF dan diterapkan langsung dalam Gini impurity CART.
- `max_depth=None` tidak berarti tree tanpa regularisasi: `min_samples_leaf=40` dan `min_samples_split=80` tetap membatasi kompleksitas.
- Threshold klasifikasi dipilih dari OOF prediction, bukan dari test/public leaderboard.
- Preprocessing selalu di-fit ulang pada training fold.
- Model final tetap **satu CART scratch**; repeated CV hanya untuk model selection, bukan averaging prediction.


<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Train-Only Audit: Pola `person_id` <a name="11"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

Bagian ini hanya menganalisis `train.csv`.

Tujuannya adalah memeriksa apakah `person_id` benar-benar identifier acak atau menyimpan struktur source-order. Label test tidak dibaca.

Jika positive rate berubah ekstrem berdasarkan rentang ID, maka ID mempunyai predictive information pada split kompetisi. Namun informasi tersebut harus dilabeli sebagai **artifact-sensitive**, bukan fitur substantif tentang risiko kredit.

In [ ]:
# ================================================================
# TRAIN-ONLY PERSON_ID STRUCTURE AUDIT
# ================================================================

id_audit_df = train_df[[ID_COL, TARGET]].copy()
id_audit_df = id_audit_df.sort_values(ID_COL).reset_index(drop=True)

print("Train rows :", len(id_audit_df))
print("ID min     :", int(id_audit_df[ID_COL].min()))
print("ID max     :", int(id_audit_df[ID_COL].max()))
print("Positive % :", f"{100 * id_audit_df[TARGET].mean():.2f}%")

# Fixed-width bins dibentuk dari rentang ID, bukan target.
ID_BIN_WIDTH = 2500
id_min = int(
    np.floor(id_audit_df[ID_COL].min() / ID_BIN_WIDTH)
    * ID_BIN_WIDTH
)
id_max = int(
    np.ceil((id_audit_df[ID_COL].max() + 1) / ID_BIN_WIDTH)
    * ID_BIN_WIDTH
)

id_edges = np.arange(
    id_min,
    id_max + ID_BIN_WIDTH,
    ID_BIN_WIDTH,
)

id_audit_df["id_bin"] = pd.cut(
    id_audit_df[ID_COL],
    bins=id_edges,
    right=False,
    include_lowest=True,
)

id_bin_summary = (
    id_audit_df.groupby("id_bin", observed=False)
    .agg(
        train_rows=(TARGET, "size"),
        positive_rate=(TARGET, "mean"),
        positive_count=(TARGET, "sum"),
        id_min=(ID_COL, "min"),
        id_max=(ID_COL, "max"),
    )
    .reset_index()
)

# Test coverage ditampilkan TANPA label test.
test_id_only = test_df[[ID_COL]].copy()
test_id_only["id_bin"] = pd.cut(
    test_id_only[ID_COL],
    bins=id_edges,
    right=False,
    include_lowest=True,
)

test_bin_count = (
    test_id_only.groupby("id_bin", observed=False)
    .size()
    .rename("test_rows")
    .reset_index()
)

id_bin_summary = id_bin_summary.merge(
    test_bin_count,
    on="id_bin",
    how="left",
)

id_bin_summary["test_rows"] = (
    id_bin_summary["test_rows"]
    .fillna(0)
    .astype(int)
)

display(
    id_bin_summary.style.format({
        "positive_rate": "{:.3f}",
    })
)

In [ ]:
plt.figure(figsize=(12, 5))

x_pos = np.arange(len(id_bin_summary))

plt.bar(
    x_pos,
    id_bin_summary["positive_rate"].to_numpy(),
)

plt.axhline(
    train_df[TARGET].mean(),
    linestyle="--",
    linewidth=1.5,
    label="Overall train positive rate",
)

plt.xticks(
    x_pos,
    id_bin_summary["id_bin"].astype(str),
    rotation=70,
    ha="right",
)

plt.ylabel("Positive rate pada train")
plt.xlabel("Rentang person_id")
plt.title("Train-only target rate berdasarkan rentang person_id")
plt.legend()
plt.tight_layout()
plt.show()

### Interpretasi audit ID

Jika beberapa blok ID mempunyai positive rate mendekati `0` atau `1`, maka `person_id` bukan identifier acak yang netral pada dataset ini.

**Penting:** notebook tidak mengubah pola tersebut menjadi aturan manual. ID hanya dimasukkan sebagai kolom numerik dan CART menentukan sendiri apakah serta di mana ia melakukan split.

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# V2+ ID-Aware CART <a name="12"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

Cabang ID-aware menggunakan class `CARTClassifierScratch` yang sama dengan V2.

Perbedaannya hanya:

```text
V2:
X = seluruh predictor selain person_id

V2+:
X = person_id + predictor lain
```

Jadi tidak ada algoritma classifier baru.

In [ ]:
# Feature set ID-aware.
# TARGET dibuang, tetapi person_id DIPERTAHANKAN sebagai predictor.
feature_cols_id_aware = [
    c for c in train_df.columns
    if c != TARGET
]

assert ID_COL in feature_cols_id_aware
assert TARGET not in feature_cols_id_aware

print("No-ID feature count   :", len(feature_cols))
print("ID-aware feature count:", len(feature_cols_id_aware))
print("ID included?          :", ID_COL in feature_cols_id_aware)

In [ ]:
# Grid dibuat cukup ringkas karena tujuan utamanya stability, bukan brute force.
ID_AWARE_CART_GRID = [
    {
        "max_depth": 8,
        "min_samples_leaf": 20,
        "positive_class_weight": 1.0,
    },
    {
        "max_depth": 10,
        "min_samples_leaf": 20,
        "positive_class_weight": 1.0,
    },
    {
        "max_depth": 12,
        "min_samples_leaf": 20,
        "positive_class_weight": 1.0,
    },
    {
        "max_depth": None,
        "min_samples_leaf": 30,
        "positive_class_weight": 1.0,
    },
    {
        "max_depth": None,
        "min_samples_leaf": 40,
        "positive_class_weight": 1.1,
    },
    {
        "max_depth": None,
        "min_samples_leaf": 40,
        "positive_class_weight": 1.2,
    },
]


def make_id_aware_cart(params, seed=SEED):
    leaf = int(params["min_samples_leaf"])
    depth = params["max_depth"]

    return CARTClassifierScratch(
        max_depth=None if depth is None else int(depth),
        min_samples_split=max(40, 2 * leaf),
        min_samples_leaf=leaf,
        max_thresholds=None,
        positive_class_weight=float(
            params["positive_class_weight"]
        ),
        random_state=seed,
    )


def run_id_aware_oof(
    params,
    cv_seed=42,
    n_splits=5,
):
    y_all = train_df[TARGET].to_numpy(dtype=int)
    oof_score = np.zeros(len(train_df), dtype=float)

    start_time = time.perf_counter()

    for fold_id, (tr_idx, va_idx) in enumerate(
        stratified_kfold_indices(
            y_all,
            n_splits=n_splits,
            random_state=cv_seed,
        ),
        start=1,
    ):
        # Fold-safe preprocessing.
        pre = ScratchTabularPreprocessor()

        X_tr = pre.fit_transform(
            train_df.iloc[tr_idx][feature_cols_id_aware]
        )
        X_va = pre.transform(
            train_df.iloc[va_idx][feature_cols_id_aware]
        )

        model = make_id_aware_cart(
            params,
            seed=cv_seed * 100 + fold_id,
        )

        model.fit(X_tr, y_all[tr_idx])
        oof_score[va_idx] = (
            model.predict_proba(X_va)[:, 1]
        )

    threshold, macro_f1_oof = (
        optimize_macro_f1_threshold(
            y_all,
            oof_score,
        )
    )

    return {
        "params": params.copy(),
        "cv_seed": cv_seed,
        "threshold": threshold,
        "macro_f1_oof": macro_f1_oof,
        "roc_auc_oof": roc_auc_score(
            y_all,
            oof_score,
        ),
        "oof_score": oof_score,
        "time_seconds": time.perf_counter() - start_time,
    }

## Phase A — 5-Fold OOF Search

Seluruh candidate dinilai hanya pada train OOF. Public leaderboard dan label eksternal tidak digunakan.

In [ ]:
ID_TUNING_SEED = 2026

id_aware_phase_a_rows = []
id_aware_cv_cache = {}

for params in ID_AWARE_CART_GRID:
    result = run_id_aware_oof(
        params,
        cv_seed=ID_TUNING_SEED,
        n_splits=5,
    )

    id_aware_cv_cache[
        (str(params), ID_TUNING_SEED)
    ] = result

    id_aware_phase_a_rows.append({
        "params": str(params),
        "macro_f1_oof": result["macro_f1_oof"],
        "roc_auc_oof": result["roc_auc_oof"],
        "threshold": result["threshold"],
        "time_seconds": result["time_seconds"],
    })

    print(
        f"{params} -> "
        f"OOF Macro F1={result['macro_f1_oof']:.6f} | "
        f"threshold={result['threshold']:.6f}"
    )

id_aware_phase_a = pd.DataFrame(
    id_aware_phase_a_rows
).sort_values(
    ["macro_f1_oof", "roc_auc_oof"],
    ascending=False,
).reset_index(drop=True)

display(
    id_aware_phase_a.style.format({
        "macro_f1_oof": "{:.6f}",
        "roc_auc_oof": "{:.6f}",
        "threshold": "{:.6f}",
        "time_seconds": "{:.2f}",
    })
)

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Repeated OOF: No-ID vs ID-Aware <a name="13"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

Tiga candidate ID-aware terbaik masuk repeated 5-fold stability audit.

Sebagai control, hasil V2 no-ID terbaik yang sudah dihitung sebelumnya tetap ditampilkan sehingga efek penambahan ID dapat diukur secara eksplisit.

In [ ]:
ID_STABILITY_SEEDS = [42, 2026, 777]

top_id_candidate_rows = (
    id_aware_phase_a
    .head(3)
    .reset_index(drop=True)
)

id_stability_rows = []

for candidate_id, row in top_id_candidate_rows.iterrows():
    params_text = row["params"]

    params = next(
        p for p in ID_AWARE_CART_GRID
        if str(p) == params_text
    )

    for cv_seed in ID_STABILITY_SEEDS:
        cache_key = (str(params), cv_seed)

        result = id_aware_cv_cache.get(cache_key)

        if result is None:
            result = run_id_aware_oof(
                params,
                cv_seed=cv_seed,
                n_splits=5,
            )

        id_stability_rows.append({
            "candidate_id": candidate_id,
            "variant": "ID-aware CART",
            "params": str(params),
            "cv_seed": cv_seed,
            "macro_f1_oof": result["macro_f1_oof"],
            "roc_auc_oof": result["roc_auc_oof"],
            "threshold": result["threshold"],
        })


id_stability_df = pd.DataFrame(
    id_stability_rows
)

id_stability_summary = (
    id_stability_df
    .groupby(
        ["candidate_id", "variant", "params"],
        as_index=False,
    )
    .agg(
        mean_macro_f1=("macro_f1_oof", "mean"),
        std_macro_f1=("macro_f1_oof", "std"),
        min_macro_f1=("macro_f1_oof", "min"),
        mean_roc_auc=("roc_auc_oof", "mean"),
        median_threshold=("threshold", "median"),
        threshold_std=("threshold", "std"),
    )
)

id_stability_summary["robust_score"] = (
    id_stability_summary["mean_macro_f1"]
    - 0.5 * id_stability_summary["std_macro_f1"]
)

id_stability_summary = (
    id_stability_summary
    .sort_values(
        ["robust_score", "mean_macro_f1"],
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    id_stability_summary.style.format({
        "mean_macro_f1": "{:.6f}",
        "std_macro_f1": "{:.6f}",
        "min_macro_f1": "{:.6f}",
        "mean_roc_auc": "{:.6f}",
        "median_threshold": "{:.6f}",
        "threshold_std": "{:.6f}",
        "robust_score": "{:.6f}",
    })
)

In [ ]:
best_id_row = id_stability_summary.iloc[0]

best_id_params = next(
    p for p in ID_AWARE_CART_GRID
    if str(p) == best_id_row["params"]
)

best_id_threshold = float(
    best_id_row["median_threshold"]
)

# V2 no-ID control berasal dari stability audit V2 di cell sebelumnya.
no_id_mean_f1 = float(
    best_stability_row["mean_macro_f1"]
)
no_id_std_f1 = float(
    best_stability_row["std_macro_f1"]
)
no_id_robust_score = (
    no_id_mean_f1
    - 0.5 * no_id_std_f1
)

variant_comparison = pd.DataFrame([
    {
        "variant": "V2 No-ID",
        "mean_macro_f1": no_id_mean_f1,
        "std_macro_f1": no_id_std_f1,
        "robust_score": no_id_robust_score,
        "threshold": best_robust_threshold,
        "params": str(best_robust_params),
    },
    {
        "variant": "V2+ ID-aware CART",
        "mean_macro_f1": float(
            best_id_row["mean_macro_f1"]
        ),
        "std_macro_f1": float(
            best_id_row["std_macro_f1"]
        ),
        "robust_score": float(
            best_id_row["robust_score"]
        ),
        "threshold": best_id_threshold,
        "params": str(best_id_params),
    },
]).sort_values(
    "robust_score",
    ascending=False,
).reset_index(drop=True)

display(
    variant_comparison.style.format({
        "mean_macro_f1": "{:.6f}",
        "std_macro_f1": "{:.6f}",
        "robust_score": "{:.6f}",
        "threshold": "{:.6f}",
    })
)

BEST_VARIANT = variant_comparison.iloc[0]["variant"]

print("Train-only selected variant :", BEST_VARIANT)
print("Best ID-aware params        :", best_id_params)
print(
    "Best ID-aware mean OOF F1  :",
    f"{best_id_row['mean_macro_f1']:.6f}",
)
print(
    "Best ID-aware threshold    :",
    f"{best_id_threshold:.6f}",
)

### Catatan metodologis

Jika `V2+ ID-aware CART` unggul sekitar beberapa poin Macro-F1 pada repeated OOF, itu membuktikan bahwa `person_id` memang membawa informasi predictive pada split dataset ini.

Namun peningkatan tersebut **tidak otomatis berarti ID adalah fitur kredit yang valid secara substantif**. Untuk write-up, sebaiknya jelaskan secara transparan bahwa:

> `person_id` menunjukkan struktur source-order yang kuat. Eksperimen ID-aware dilakukan sebagai sensitivity/competition analysis. Generalisasi ke dataset dengan pengacakan ID baru tidak dapat diasumsikan sama.

Dengan penjelasan tersebut, kita tidak menyamarkan artifact sebagai insight domain.

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Final Training & Submission <a name="14"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

Notebook menghasilkan **dua submission**:

1. V2 no-ID — baseline metodologis.
2. V2+ ID-aware — single CART yang memasukkan `person_id`.

File `submission_best_scratch_v2_plus.csv` otomatis mengikuti variant dengan train-only `robust_score` tertinggi.

Jika aturan kelas/dosen kemudian melarang identifier sebagai predictor, gunakan file no-ID.

In [ ]:
y_full = train_df[TARGET].to_numpy(dtype=int)

# ================================================================
# A. FINAL V2 NO-ID
# ================================================================
no_id_pre = make_robust_preprocessor(
    best_robust_model_name
)

X_full_no_id = no_id_pre.fit_transform(
    train_df[feature_cols]
)

X_test_no_id = no_id_pre.transform(
    test_df[feature_cols]
)

no_id_model = make_robust_model(
    best_robust_model_name,
    best_robust_params,
    seed=SEED,
)

no_id_model.fit(
    X_full_no_id,
    y_full,
)

no_id_score = get_model_score(
    no_id_model,
    X_test_no_id,
)

no_id_prediction = (
    no_id_score >= best_robust_threshold
).astype(int)


# ================================================================
# B. FINAL V2+ ID-AWARE CART
# ================================================================
id_pre = ScratchTabularPreprocessor()

X_full_id = id_pre.fit_transform(
    train_df[feature_cols_id_aware]
)

X_test_id = id_pre.transform(
    test_df[feature_cols_id_aware]
)

id_model = make_id_aware_cart(
    best_id_params,
    seed=SEED,
)

id_model.fit(
    X_full_id,
    y_full,
)

id_score = (
    id_model.predict_proba(
        X_test_id
    )[:, 1]
)

id_prediction = (
    id_score >= best_id_threshold
).astype(int)


print(
    "No-ID predicted positive :",
    f"{100 * no_id_prediction.mean():.2f}%"
)
print(
    "ID-aware positive        :",
    f"{100 * id_prediction.mean():.2f}%"
)
print(
    "Different predictions    :",
    int(np.sum(no_id_prediction != id_prediction)),
    "rows",
)

In [ ]:
submission_no_id = pd.DataFrame({
    ID_COL: test_df[ID_COL].to_numpy(),
    TARGET: no_id_prediction.astype(int),
})

submission_id_aware = pd.DataFrame({
    ID_COL: test_df[ID_COL].to_numpy(),
    TARGET: id_prediction.astype(int),
})

submission_no_id_path = (
    OUTPUT_DIR
    / "submission_v2_no_id_weighted_cart.csv"
)

submission_id_aware_path = (
    OUTPUT_DIR
    / "submission_v2_plus_id_aware_cart.csv"
)

submission_no_id.to_csv(
    submission_no_id_path,
    index=False,
)

submission_id_aware.to_csv(
    submission_id_aware_path,
    index=False,
)

if BEST_VARIANT == "V2+ ID-aware CART":
    best_submission = submission_id_aware.copy()
    FINAL_SUBMISSION_VARIANT = "V2+ ID-aware CART"
else:
    best_submission = submission_no_id.copy()
    FINAL_SUBMISSION_VARIANT = "V2 No-ID"

best_submission_path = (
    OUTPUT_DIR
    / "submission_best_scratch_v2_plus.csv"
)

best_submission.to_csv(
    best_submission_path,
    index=False,
)

for submission in [
    submission_no_id,
    submission_id_aware,
    best_submission,
]:
    assert list(submission.columns) == list(
        sample_submission.columns
    )
    assert len(submission) == len(test_df)
    assert np.array_equal(
        submission[ID_COL].to_numpy(),
        test_df[ID_COL].to_numpy(),
    )
    assert set(
        submission[TARGET].unique()
    ).issubset({0, 1})

print("No-ID file    :", submission_no_id_path)
print("ID-aware file :", submission_id_aware_path)
print("AUTO best     :", best_submission_path)
print("AUTO variant  :", FINAL_SUBMISSION_VARIANT)

display(best_submission.head(10))

## Kaggle Submission

Untuk mengejar skor kompetisi berdasarkan train-only OOF, submit:

```text
submission_best_scratch_v2_plus.csv
```

Untuk versi yang tidak menggunakan identifier, submit:

```text
submission_v2_no_id_weighted_cart.csv
```

In [ ]:
SUBMIT_TO_KAGGLE = False
COMPETITION_NAME = "ai-lab-recruitment-task-2"

SUBMISSION_MESSAGE = (
    f"V2+ single scratch CART | {FINAL_SUBMISSION_VARIANT} | "
    "repeated OOF selection"
)

if SUBMIT_TO_KAGGLE:
    import subprocess

    command = [
        "kaggle",
        "competitions",
        "submit",
        "-c",
        COMPETITION_NAME,
        "-f",
        str(best_submission_path),
        "-m",
        SUBMISSION_MESSAGE,
    ]

    completed = subprocess.run(
        command,
        text=True,
        capture_output=True,
    )

    print(completed.stdout)

    if completed.returncode != 0:
        print(completed.stderr)
else:
    print(
        "SUBMIT_TO_KAGGLE=False. "
        "File submission sudah dibuat."
    )

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Kesimpulan <a name="15"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

V2+ mempertahankan desain V2 dan hanya menambahkan satu eksperimen terkontrol.

1. Tiga algoritma wajib tetap dibuat from scratch.
2. V2 Weighted CART no-ID tetap menjadi baseline.
3. ID-aware branch tetap memakai **satu CART from scratch**.
4. Tidak ada manual label override berdasarkan ID.
5. Hyperparameter dan threshold dipilih dari train-only OOF.
6. Test label/public leaderboard tidak menjadi objective tuning.
7. Kedua submission disimpan terpisah agar penggunaan identifier dapat dibedakan secara transparan.
8. Jika ID-aware unggul, kesimpulan yang benar adalah bahwa **split dataset mengandung source-order signal**, bukan bahwa `person_id` merupakan faktor kredit yang substantif.

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

# Optional Post-Submission Audit <a name="16"></a>

<hr style="border: 2px solid #8E7B6B; margin-top: 10px;">

Evaluasi eksternal **OFF secara default**.

Bagian ini membandingkan V2 no-ID dan V2+ ID-aware setelah submission/configuration dibekukan. Jangan gunakan hasil audit untuk melakukan iterasi hyperparameter baru jika ingin mempertahankan evaluasi yang fair.

In [ ]:
EVALUATE_EXTERNAL_LABELS = False

EXTERNAL_LABELS_KAGGLE_PATH = Path(
    "/kaggle/input/datasets/kurtmikhael/"
    "external-labels/test_labels.csv"
)

EXTERNAL_LABELS_LOCAL_PATH = Path(
    "/mnt/data/test_labels.csv"
)

EXTERNAL_LABELS_PATH = (
    EXTERNAL_LABELS_KAGGLE_PATH
    if EXTERNAL_LABELS_KAGGLE_PATH.exists()
    else EXTERNAL_LABELS_LOCAL_PATH
)

if not EVALUATE_EXTERNAL_LABELS:
    print(
        "Evaluasi eksternal OFF. "
        "Aktifkan hanya setelah model sudah dibekukan/submitted."
    )
else:
    if not EXTERNAL_LABELS_PATH.exists():
        raise FileNotFoundError(
            "Label eksternal tidak ditemukan."
        )

    external_labels = pd.read_csv(
        EXTERNAL_LABELS_PATH
    )[[ID_COL, TARGET]].copy()

    audit_rows = []

    audit_submissions = {
        "V2 No-ID": submission_no_id,
        "V2+ ID-aware CART": submission_id_aware,
        "AUTO selected": best_submission,
    }

    for name, submission in audit_submissions.items():
        merged = submission.merge(
            external_labels,
            on=ID_COL,
            how="inner",
            suffixes=("_prediction", "_truth"),
            validate="one_to_one",
        )

        y_truth = merged[
            f"{TARGET}_truth"
        ].to_numpy(dtype=int)

        y_pred = merged[
            f"{TARGET}_prediction"
        ].to_numpy(dtype=int)

        audit_rows.append({
            "variant": name,
            "rows": len(merged),
            "macro_f1": f1_score(
                y_truth,
                y_pred,
                average="macro",
            ),
            "accuracy": accuracy_score(
                y_truth,
                y_pred,
            ),
        })

    audit_summary = pd.DataFrame(
        audit_rows
    ).sort_values(
        "macro_f1",
        ascending=False,
    )

    display(
        audit_summary.style.format({
            "macro_f1": "{:.6f}",
            "accuracy": "{:.6f}",
        })
    )

    # Classification report untuk model AUTO.
    auto_merged = best_submission.merge(
        external_labels,
        on=ID_COL,
        how="inner",
        suffixes=("_prediction", "_truth"),
        validate="one_to_one",
    )

    y_truth = auto_merged[
        f"{TARGET}_truth"
    ].to_numpy(dtype=int)

    y_pred = auto_merged[
        f"{TARGET}_prediction"
    ].to_numpy(dtype=int)

    print("POST-SUBMISSION AUDIT — AUTO MODEL")
    print("Variant :", FINAL_SUBMISSION_VARIANT)
    print(
        "Macro F1:",
        f"{f1_score(y_truth, y_pred, average='macro'):.6f}",
    )
    print(
        "Accuracy:",
        f"{accuracy_score(y_truth, y_pred):.6f}",
    )
    print()
    print(
        classification_report(
            y_truth,
            y_pred,
            digits=4,
        )
    )